# Guide Eye: train the door/stairs model on Colab

Trains `models/door_stairs.pt` on a free Colab GPU, so the laptop's RAM and disk are not used.

1. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
2. Keep this tab open (free Colab disconnects idle sessions). Training takes a few hours.
3. Progress is saved to Google Drive (`MyDrive/guide_eye_training`). If the session disconnects,
   just **Run all** again: the images are downloaded again and training continues from the last epoch.
4. The last cell downloads the new model if it beat the old one. Put it in `models/` on the laptop.

In [ ]:
!nvidia-smi -L

In [ ]:
# Drive keeps the training progress and the filtered annotation lists across sessions
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE = '/content/drive/MyDrive/guide_eye_training'
os.makedirs(f'{SAVE}/runs', exist_ok=True)
os.makedirs(f'{SAVE}/openimages_raw', exist_ok=True)

if not os.path.exists('/content/mini-project'):
    !git clone -q https://github.com/rajsingh1301/mini-project.git /content/mini-project
%cd /content/mini-project
!mkdir -p datasets && ln -sfn {SAVE}/runs runs && ln -sfn {SAVE}/openimages_raw datasets/openimages_raw
!pip install -q ultralytics

In [ ]:
# All ~17k Open Images door/stairs photos (about 10 minutes; the first run also filters the annotation list)
!python training/prepare_dataset.py

In [ ]:
# Continues from the last saved epoch after a disconnect.
# If training already finished in an earlier session, skip this cell.
if os.path.exists('runs/door_stairs/weights/last.pt'):
    !python training/train.py --resume 2>&1 | tee -a runs/train_colab.log
else:
    !python training/train.py --cache disk 2>&1 | tee -a runs/train_colab.log

In [ ]:
# Test-split scores of the installed model, and the download if the new model won
!grep -E "^(door|stairs|all) |Installed|kept the installed" runs/train_colab.log | tail -5

import filecmp
from google.colab import files
best = 'runs/door_stairs/weights/best.pt'
if filecmp.cmp(best, 'models/door_stairs.pt', shallow=False):
    print('The new model scored higher and is installed. Downloading it...')
    !cp models/door_stairs.pt {SAVE}/door_stairs.pt
    files.download('models/door_stairs.pt')
else:
    print('The new model did not beat the current one; keep the laptop model.')